# 01 · Evaluate the pretrained ACT checkpoint (Step 1b)

<a href="https://colab.research.google.com/github/danielamrh/act-visual-robustness/blob/main/notebooks/01_eval_pretrained.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Sanity check for the evaluation pipeline before training anything ourselves.
The Hugging Face checkpoint [`lerobot/act_aloha_sim_transfer_cube_human`](https://huggingface.co/lerobot/act_aloha_sim_transfer_cube_human)
reports **83.0 % success over 500 episodes** on `AlohaTransferCube-v0`.
If our number lands inside that ballpark (mind the confidence interval), the pipeline is trustworthy.

**Runtime → Change runtime type → T4 GPU** before running.

## 0 · Environment check

In [ ]:
import sys, subprocess
print(sys.version)
assert sys.version_info >= (3, 12), "LeRobot 0.6.x needs Python >= 3.12"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU - switch the runtime to T4!")

## 1 · Drive + repo + install
Code comes from GitHub (always fresh via `git pull`), outputs go to Google Drive.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

REPO = "act-visual-robustness"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/danielamrh/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
COMMIT = !git rev-parse --short HEAD
COMMIT = COMMIT[0]
print("commit:", COMMIT)

# labmaze (pulled in by dm-control) has no wheel for Python >= 3.13 and needs Bazel
# to build. gym-aloha never uses it, so install an empty placeholder first.
if sys.version_info >= (3, 13):
    !pip install -q ./tools/labmaze_stub

# installs lerobot[aloha] pinned in pyproject.toml (takes a few minutes)
!pip install -q -e ".[sim]"
sys.path.insert(0, f"{REPO_DIR}/src")  # editable install is only picked up after a restart

# GPU rendering: without NVIDIA's EGL registration MuJoCo silently renders on the CPU
from avr.colab import ensure_nvidia_egl, gl_renderer
ensure_nvidia_egl()
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
renderer = gl_renderer()
print("OpenGL renderer:", renderer)
if "llvmpipe" in renderer.lower() or renderer.startswith("failed"):
    print("⚠ MuJoCo is NOT rendering on the GPU - rollouts will be extremely slow")

DRIVE_ROOT = "/content/drive/MyDrive/act_robustness"
os.makedirs(DRIVE_ROOT, exist_ok=True)

### Quick check: can we create and render the ALOHA env?

In [ ]:
import gymnasium as gym
import gym_aloha  # registers gym_aloha/* envs
import matplotlib.pyplot as plt

import time
env = gym.make("gym_aloha/AlohaTransferCube-v0")
obs, _ = env.reset(seed=0)
img = obs["top"] if isinstance(obs, dict) and "top" in obs else env.render()

# speed check: env step incl. observation rendering, zero actions
t0 = time.perf_counter()
for _ in range(50):
    env.step(env.action_space.sample() * 0)
dt = (time.perf_counter() - t0) / 50
env.close()
print(f"{dt*1000:.0f} ms per env step -> one 400-step episode ≈ {400*dt:.0f} s (+ video rendering)")
plt.imshow(img); plt.axis("off"); plt.title("AlohaTransferCube-v0, seed 0");

## 2 · Migrate the checkpoint to the current LeRobot format
The Hub checkpoint predates LeRobot's processor pipelines (normalization used to live inside the model).
LeRobot ships a migration script for exactly this; we run it once and cache the result on Drive.

In [ ]:
import lerobot, pathlib
print("lerobot", lerobot.__version__)

PRETRAINED = "lerobot/act_aloha_sim_transfer_cube_human"
POLICY_DIR = f"{DRIVE_ROOT}/checkpoints/act_aloha_sim_transfer_cube_human_migrated"
migrate_script = pathlib.Path(lerobot.__file__).parent / "processor" / "migrate_policy_normalization.py"

if not os.path.exists(f"{POLICY_DIR}/config.json"):
    !python {migrate_script} --pretrained-path {PRETRAINED} --output-dir {POLICY_DIR}
!ls {POLICY_DIR}

## 3 · Smoke test (10 episodes)
Quick check that env, rendering and policy work together before spending GPU time.

Evaluations run as **background jobs** that only write to a log file; the cell prints one status line
per interval (elapsed time, free RAM, the job's RAM, latest progress). Streaming LeRobot's progress bars
into the cell crashed the browser tab on long runs.

In [ ]:
import time
from avr.eval.chunked import run_chunked_eval

EVAL_BASE = f"{DRIVE_ROOT}/eval/pretrained_transfer_cube"
smoke_root = f"{EVAL_BASE}/smoke_{COMMIT}_{time.strftime('%Y%m%d-%H%M%S')}"
run_chunked_eval(POLICY_DIR, smoke_root, n_episodes=10, chunk_size=10, interval=15);

## 4 · Full evaluation (500 episodes, resumable)
500 episodes give a ±~3.5 % confidence interval, like the model card. They run in **chunks of 100**,
each a separate process with its own seed range (1000-1099, 1100-1199, ...). Finished chunks stay on Drive:
if the session dies, re-run sections 0–2 and this cell, and it continues with the missing chunks.
Each chunk takes roughly 10 minutes.

In [ ]:
N_EPISODES = 500
EVAL_ROOT = f"{EVAL_BASE}/n{N_EPISODES}_seed1000"
info_paths = run_chunked_eval(POLICY_DIR, EVAL_ROOT, n_episodes=N_EPISODES, chunk_size=100, seed=1000)

## 5 · Summary: success rate, confidence interval, failure stages
Works on partial results too (e.g. 3 of 5 chunks done).

In [ ]:
import glob, json
from avr.eval.chunked import chunk_info_paths
from avr.eval.stats import merge_eval_infos, summarize_eval, format_summary

EVAL_ROOT = f"{EVAL_BASE}/n500_seed1000"
info_paths = chunk_info_paths(EVAL_ROOT)
print(f"{len(info_paths)} finished chunks in {EVAL_ROOT}")
summary = summarize_eval(merge_eval_infos(info_paths))
print(format_summary(summary))
print("\nReference (model card): 83.0 % over 500 episodes")

with open(f"{EVAL_ROOT}/summary.json", "w") as f:
    json.dump({"commit": COMMIT, "checkpoint": PRETRAINED, "chunks": len(info_paths), **summary}, f, indent=2)

## 6 · Watch a rollout

In [ ]:
from IPython.display import Video
videos = sorted(glob.glob(f"{EVAL_ROOT}/**/*.mp4", recursive=True))
print(len(videos), "videos")
Video(videos[0], embed=True, width=480) if videos else None